<a href="https://colab.research.google.com/github/sngillard/student-outcome-classification/blob/main/notebooks/00_run_in_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Student Success Prediction System for First-Year College Students**

This notebook demonstrates the Gradient Boosting Classifier Model for predicting dropout risk of first-year college students using socioeconomic, demographic, and academic performance data.


**Possible predicted student outcomes:**
*   **0**: Dropout
*   **1**: Remain Enrolled
*   **2**: Graduate

**Instructions to run:**
1. Click Runtime -> Run all.
2. ***Scroll to the bottom to enter student information.***
3. View the predicted outcome.

In [ ]:
!pip -q install scikit-learn ipywidgets pandas numpy

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import ipywidgets as widgets
from IPython.display import display, clear_output


In [ ]:
# Load processed dataset from GitHub
processed_data_url = 'https://raw.githubusercontent.com/sngillard/student-outcome-classification/refs/heads/main/data/processed/student_data_processed_2026-01-08.csv'

df = pd.read_csv(processed_data_url)

print('Dataset Shape:', df.shape)
display(df.head())

In [ ]:
# Split dataset into features and target
X = df.drop(columns=['Target'])
y = df['Target']

# Split data into training and testing sets with 80/20 split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print('Training shape:', X_train.shape)
print('Testing shape:', X_test.shape)

**NOTE:**
*   Hyperparameter tuning using RandomizedSearchCV was performed in the 04_gradient_boosting_model.ipynb notebook to identify optimal model parameters.
*   To improve runtime and user experience, the best parameters found in the IntelliJ Jupyter notebook are hard-coded below and used to train the final model.
*   An optional hyperparameter tuning block is included in this notebook below and can be enabled by setting run_tuning to True. However, this increases run-time from under one minute to about 2-3 minutes for the user.

In [ ]:
# Best params from RandomizedSearchCV (from 04_gradient_boosting_model.ipynb notebook output)
# These are hard-coded parameters for faster runtime in the interactive user interface in notebook 00_run_in_colab.ipynb
set_best_params = {
    'learning_rate': 0.1707829063523625,
    'max_depth': 3,
    'n_estimators': 143,
    'subsample': np.float64(0.9480528898228043),
    'random_state': 42
}

# Train the Gradient Boosting model using the optimized parameters
gb_classifier_model_set_best_params = GradientBoostingClassifier(**set_best_params)
gb_classifier_model_set_best_params.fit(X_train, y_train)

# Default model (used later unless run_tuning is set to True below)
final_classifier_model = gb_classifier_model_set_best_params

# Evaluate model performance on test set using accuracy, classification report, and confusion matrix
y_prediction_gb = gb_classifier_model_set_best_params.predict(X_test)
print('Test accuracy of prediction model:', accuracy_score(y_test, y_prediction_gb))
print('\nClassification Report:\n', classification_report(y_test, y_prediction_gb))
print('\nConfusion Matrix:\n', confusion_matrix(y_test, y_prediction_gb))

**Note**
*   Optional code below will rerun hyperparamter tuning in the Colab notebook (slow, about 2-3 minutes runtime).
*   To use the hard-coded values above, leave **run_tuning** set to false and engage with the UI using the hard-coded hyperparameter values.



In [ ]:
# Default model uses hard-coded parameter values for increased processing speed for the user.
final_classifier_model = gb_classifier_model_set_best_params

# Set Run Tuning to true if you want to re-run hyperparameter tuning, but this may take 2-3 minutes, opposed to the faster performance when running the model with hard-coded best parameters found from running RandomizedSearchCV in IntelliJ.
run_tuning = False

if run_tuning:
  from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
  from scipy.stats import randint, uniform


# Build Gradient Boosting Classifier model
  gb_classifier_base_model = GradientBoostingClassifier(random_state=42)

# Run hyperparameter tuning for the model
  param_distributions_gb = {
      'n_estimators': randint(80, 200),
      'learning_rate': uniform(0.03, 0.15),
      'max_depth': randint(2, 5),
      'subsample': uniform(0.7, 0.25),
  }

# Stratified cross-validation
  cross_val = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Randomized hyperparameter search
  random_search_gb = RandomizedSearchCV(
      estimator= gb_classifier_base_model,
      param_distributions=param_distributions_gb,
      n_iter=10,
      scoring='accuracy',
      cv = cross_val,
      random_state=42,
      n_jobs = -1,
      verbose=1
  )

# Fit model with best parameters from RandomSearchCV
  random_search_gb.fit(X_train, y_train)

  print('Best gradient boosting params:', random_search_gb.best_params_)
  print('Best gradient boosting cross-validation accuracy:', random_search_gb.best_score_)

# Overwrite the default model if run_tuning is set to True. final_classifier_model uses the tuned model if run_tuning is set to True and uses the hard-coded model above if run_tuning is set to False.
  final_classifier_model = random_search_gb.best_estimator_

# Evaluate tuned model on the test set
y_prediction_gb = final_classifier_model.predict(X_test)
print('Test accuracy of final model:', accuracy_score(y_test, y_prediction_gb))


**The interactive user prototype below uses 6 of the 34 features included in the dataset.**
*   These features were selected based on the EDA findings performed in 02_exploratory_data_analysis.ipynb.
*   1st semester academic performnace variables (grade, approved, and enrolled) were selected as they showed the strongest correlation with student outcomes during EDA. They also provide a realistic prediciton data point before second semester data is available. This helps identify at-risk students before the second semester begins.
*   Tuition fees up to date, Scholarship holder, and Displaced were also selected for the UI as they showed stronger corrleations to the target variable.
*   The features that were not selected to be used in the user interface are set to zero so the selected variables only are used in the prediction user interface.





In [ ]:
from sklearn.preprocessing import StandardScaler

# Load raw dataset for the UI because the user will be entering values in the form that the values are presented in the raw dataset (not scaled form like the model uses... the model will scale what the user enters using StandardScaler())
raw_data_url = ('https://raw.githubusercontent.com/sngillard/student-outcome-classification/refs/heads/main/data/raw/student_data_raw.csv')
df_raw = pd.read_csv(raw_data_url)

# Fix the typo in the raw dataset
df_raw = df_raw.rename(columns={'Nacionality': 'Nationality'})

# Map the Target variable in the raw dataset to its numerical value as found in the processed dataset
target_variable_map = {
    'Dropout': 0,
    'Enrolled': 1,
    'Graduate': 2
}
if df_raw['Target'].dtype == 'object':
  df_raw['Target'] = df_raw['Target'].map(target_variable_map)

# Split the data into Target and feature variables
X_raw = df_raw.drop(columns=['Target'])
y = df_raw['Target']

# Split raw data into 80/20 train/test split
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size = 0.2, stratify = y, random_state = 42
)

# Fit StandardScaler() on training data
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_raw), columns = X_train_raw.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test_raw), columns = X_test_raw.columns)

# Hard code the best parameters for the model (found using GridSearchCV)
set_best_params = {
    'learning_rate': np.float64(0.1707829063523625),
    'max_depth': 3,
    'n_estimators': 143,
    'subsample': np.float64(0.9480528898228043),
    'random_state': 42
}

final_classifier_model = GradientBoostingClassifier(**set_best_params)
final_classifier_model.fit(X_train_scaled, y_train)

# Evaluate the model on the test set
y_test_prediction = final_classifier_model.predict(X_test_scaled)

test_accuracy = accuracy_score(y_test, y_test_prediction)
test_confusion_matrix = confusion_matrix(y_test, y_test_prediction)

# Print report for user with precision, recall, F1-score for class and averages
generate_test_report = classification_report (
    y_test, y_test_prediction, target_names = ['Dropout', 'Remain Enrolled', 'Graduate'], digits = 2
)

# Print model accuracy
print('Train accuracy:', final_classifier_model.score(X_train_scaled, y_train))
print('Test accuracy:', final_classifier_model.score(X_test_scaled, y_test))
raw_feature_columns = list(X_train_raw.columns)

# Map the classes to labels (0 is dropout, 1 enrolled, 2 graduate)
label_map = {0: 'Dropout', 1: 'Remain Enrolled', 2: 'Graduate'}

# The features included as user inputs (selected based on EDA performed in notebook 02_exploratory_data_analysis.ipynb)
ui_selected_features = [
    'Curricular units 1st sem (grade)',
    'Curricular units 1st sem (approved)',
    'Curricular units 1st sem (enrolled)',
    'Tuition fees up to date',
    'Scholarship holder',
    'Displaced'
]

feature_widgets = {
    'Curricular units 1st sem (grade)': widgets.FloatSlider(
        value = 10.0,
        min = 0.0,
        max = 20.0,
        step = 0.5,
        description = '1st Semester Curricular Units (Grade)',
        style = {'description_width': '320px'},
        layout = widgets.Layout(width = '95%'),
        continuous_update = False
    ),

      'Curricular units 1st sem (approved)': widgets.IntSlider(
        value = 5,
        min = 0,
        max = 26,
        step = 1,
        description = '1st Semester Curricular Units (Approved)',
        style = {'description_width': '320px'},
        layout = widgets.Layout(width = '95%'),
        continuous_update = False
      ),

    'Curricular units 1st sem (enrolled)': widgets.IntSlider(
        value = 6,
        min = 0,
        max = 26,
        step = 1,
        description = '1st Semester Curricular Units (Enrolled)',
        style = {'description_width': '320px'},
        layout = widgets.Layout(width = '95%'),
        continuous_update = False
    ),

    'Tuition fees up to date': widgets.RadioButtons(
        options = [('No', 0), ('Yes', 1)],
        value = 0,
        description = 'Tuition Paid (Y/N)',
        style = {'description_width': '320px'},
        layout = widgets.Layout(width = '90%')
    ),

    'Scholarship holder': widgets.RadioButtons(
        options = [('No', 0), ('Yes', 1)],
        value = 0,
        description = 'Scholarship (Y/N)',
        style = {'description_width': '320px'},
        layout = widgets.Layout(width = '90%')
    ),

    'Displaced': widgets.RadioButtons(
        options = [('No', 0), ('Yes', 1)],
        value = 0,
        description = 'Displaced (Y/N)',
        style = {'description_width': '320px'},
        layout = widgets.Layout(width = '90%')
    )
}

# Toggle to show or hide evaluation metrics when printing the prediction outcome
show_evaluation_metrics = widgets.Checkbox(
    value = True,
    description = 'Show model evaluation metrics on the test set',
    indent = False
)

# Build the user interface to display to user
project_title = widgets.HTML('<h2>Student Outcome Predictor for First-Year College Students<h2>')

subtitle = widgets.HTML('<h3>Gradient Boosting Classifier Model</h3>')

user_instructions = widgets.HTML('<p>Adjust inputs, then click <b>Predict Outcome</b><p>')

# UI uses vertical layout
ui = widgets.VBox([
    feature_widgets[col] for col in ui_selected_features
])

# Predict button for user
predict_button = widgets.Button(
    description = 'Predict Outcome',
    button_style = 'primary'
)
output = widgets.Output()

display(project_title, subtitle, user_instructions, ui, show_evaluation_metrics, predict_button, output)

# Build a raw data row with the user input values, then scale these values using StandardScaler, then make the prediction based on the scaled input values

# The non-selected features from the dataset are set to 0 (the mean of the scaled z-score values is zero), and the six values for the feature widgets are added then transformed using standardScaler().
def build_input_row_raw():
  input_data = X_train_raw.mean().to_dict()
  for col in ui_selected_features:
    input_data[col] = feature_widgets[col].value
  return pd.DataFrame([input_data], columns = X_train_raw.columns)

# Scale the selected features (ui_selected_features)
def build_input_row_scaled():
  raw_row = build_input_row_raw()
  scaled_data_array = scaler.transform(raw_row)
  return pd.DataFrame(scaled_data_array, columns = raw_feature_columns)

# Predict students' drop-out risk
def on_predict_button_clicked(b):
    with output:
      # Clear previous output
      output.clear_output()

      # Convert the user input from raw to scaled values
      input_scaled = build_input_row_scaled()

      prediction_class = int(final_classifier_model.predict(input_scaled)[0])
      prediction_label = label_map.get(prediction_class, f'Unknown ({prediction_class})')

      # Print prediction outcome based on user input
      print(f'')
      print(f'Predicted Student Outcome:')
      print(f'{prediction_label}')

      # Show model evaluation metrics
      if show_evaluation_metrics.value:
        print('')
        print(f'MODEL EVALUATION FOR TEST SET')
        print(f'Accuracy: {test_accuracy:.2f}')
        print(generate_test_report)
        print(f'Note:')
        print(f'-Accuracy is the percentage of predictions the system correctly makes with the test set.')
        print(f'-Precision score reflects how often the model is correct when predicting an outcome on the test set.')
        print(f'-Recall score reflects how well the model finds all students belonging to an outcome class in the test set.')
        print(f'-F1-score balances precision and recall scores (where 1 is a perfect score and 0 is the worst score).')

predict_button.on_click(on_predict_button_clicked)